#### **OPTIMALIDAD DE LA SOLUCION - ALGORITMOS GREEDY**

In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
from importlib import reload

import Clases.asignacion as asignacion_module
reload(asignacion_module)
from Clases.asignacion import Asignacion

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

import Clases.producto as producto_module
reload(producto_module)
from Clases.producto import Producto

import Clases.solucion as solucion_module
reload(solucion_module)
from Clases.solucion import Solucion

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-finales/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-finales/procurement_cajas.csv")
factibilidad = pd.read_csv("Factibilidad/factibilidad_5mm.csv")

Empecemos nuevamente guardando los productos y tipos de cajas en listas.

In [2]:
caja_compras_merge = especificaciones_cajas.merge(procurement_cajas,
                                                  on="caja_tipo_id")

cajas = [
    Caja(
        caja_id = row["caja_tipo_id"],
        dim_interior_ancho = row["caja_interior_ancho"],
        dim_interior_largo = row["caja_interior_largo"],
        dim_interior_alto = row["caja_interior_alto"],
        costo_unitario = row['costo_unitario_base']
    )
    for _, row in caja_compras_merge.iterrows()
]

cajas[:5]

[<Caja 02cf77de65b70dd77905e2e33d78478f | Int: 296.0 x 395.0 x 291.0mm | Compra Total: 0>,
 <Caja 082c1cdb42b1abd201403ca33ca11ef0 | Int: 248.0 x 383.0 x 189.0mm | Compra Total: 0>,
 <Caja 0835ff365412a67b720a19713ec250f3 | Int: 286.0 x 386.0 x 278.0mm | Compra Total: 0>,
 <Caja 0b72571a5bb7429ce7de424547e8d27d | Int: 286.0 x 386.0 x 174.0mm | Compra Total: 0>,
 <Caja 10c5f9edbe2c87186bcdeb991fe8d902 | Int: 252.0 x 380.0 x 228.0mm | Compra Total: 0>]

In [3]:
prod_op_merge = catalogo_productos.merge(operaciones_planta, on="codigo_producto")

productos = [
    Producto(
        codigo_producto = row['codigo_producto'],
        cantidad_paquetes = row['cantidad_paquetes'],
        peso_paquete = row['peso_neto_paquete'],
        demanda_buenos_aires = row['volumen_producto_planta_buenos_aires'],
        demanda_curitiba = row['volumen_producto_planta_curitiba'],
        demanda_santiago = row['volumen_producto_planta_santiago'],
        demanda_monterrey = row['volumen_producto_planta_monterrey'],
        demanda_bakersfield = row['volumen_producto_planta_bakersfield'],
        dim_producto_ancho = row['dim_producto_ancho'], 
        dim_producto_largo = row['dim_producto_largo'],
        dim_producto_alto = row['dim_producto_alto']
    )
    for _, row in prod_op_merge.iterrows()
]

productos[:5]

[<Producto BR0001 | Dim Prod: 286.0 x 386.0 x 303.0mm | Demanda Total: 1546613>,
 <Producto BR0002 | Dim Prod: 296.0 x 395.0 x 260.0mm | Demanda Total: 139211>,
 <Producto BR0003 | Dim Prod: 288.0 x 388.0 x 164.0mm | Demanda Total: 172506>,
 <Producto BR0004 | Dim Prod: 296.0 x 395.0 x 224.0mm | Demanda Total: 271715>,
 <Producto BR0005 | Dim Prod: 286.0 x 386.0 x 253.0mm | Demanda Total: 7586>]

Agregamos también las cajas asignables por producto en la lista de productos.

In [4]:
# Crear un diccionario de cajas por producto desde el CSV de factibilidad
cajas_por_producto = {}
for _, row in factibilidad.iterrows():
    codigo = row['codigo_producto']
    cajas_str = row['cajas_asignables_id']
    
    if isinstance(cajas_str, str) and cajas_str:
        # Separar por '; ' y limpiar
        cajas_list = [c.strip() for c in cajas_str.split(';') if c.strip()]
        cajas_por_producto[codigo] = cajas_list

# Recorrer la lista de productos en orden y agregar las cajas
for producto in productos:
    if producto.codigo_producto in cajas_por_producto:
        for caja_id in cajas_por_producto[producto.codigo_producto]:
            producto.agregar_caja_asignable(caja_id)

#### **Greedy 0: Costos base sin descuentos**

Inicialmente, plantearemos una solución únicamente comparando los costos unitarios base, sin considerar todavía los descuentos por volúmenes anuales.

Como no hay ninguna restricción sobre la cantidad de cajas que podemos comprar de cada tipo, el problema se reduce en encontrar para cada producto el tipo de caja que más le convenga, es decir, el que ofrezca el menor costo.

Empecemos eligiendo un grosor de 5mm para todos los tipos de cajas, de manera que la restricción de headspace no supone ningún problema, pues las dimensiones internas de las cajas se diferencian hasta un 10% de las originales.

In [5]:
for caja in cajas:
    caja.elegir_grosor(5)

In [6]:
def buscar_caja_por_id(id):
    caja_a_buscar = None
    for caja in cajas:
        if caja.caja_id == id:
            caja_a_buscar = caja
    return caja_a_buscar

solucion_base = Solucion(grosor=5)

for producto in productos:
    for caja_id in producto.cajas_asignables:
        caja = buscar_caja_por_id(caja_id)
        asignacion = Asignacion(producto, caja)
        solucion_base.agregar_asignacion(asignacion, descuentos=False)

solucion_base.exportar_submmit(nombre_csv="1-base_5mm")
solucion_base.resumen_por_asignacion()

,codigo_producto,demanda_total,caja_id,utilizacion_pallet,utilizacion_caja,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,02cf77de65b70dd77905e2e33d78478f,0.323810,0.983137,927967.80,103109,15466350,16394317.80
1,BR0001,1546613,5e55b0f0e0122b55ba8d4d91fb921c5a,0.342702,0.927148,1005298.45,103109,15466350,16471648.45
2,BR0001,1546613,60eb77c934f99a121f410ec2037a2a46,0.339947,0.934944,927967.80,103109,15466350,16394317.80
3,BR0001,1546613,845990194eeb50cd131954a55fecd655,0.436688,0.972033,927967.80,77333,11599950,12527917.80
4,BR0001,1546613,8f26468860664ecfed551104179d7f3e,0.889973,0.952830,1005298.45,38667,5800050,6805348.45
...,...,...,...,...,...,...,...,...,...
5893,BR0421,145803,a7d151b83674a2f9c36356597a93821c,0.811290,0.997536,102062.10,2604,390600,492662.10
5894,BR0421,145803,d85b31778b741c8bdb462d9f332e56d2,0.834623,0.968591,94771.95,2604,390600,485371.95
5895,BR0421,145803,ea6403e25e3a52198dad19251c207a47,0.844404,0.956821,102062.10,2604,390600,492662.10
5896,BR0421,145803,ef2c4cbdd2be88c4d7a73fe0a429c842,0.817629,0.989366,102062.10,2604,390600,492662.10


#### **Greedy 1: Elección por menor costo**

Greedy (1)

Ordenamos los productos de menor a mayor según la cantidad de tipos de cajas asignables.
Iteramos los productos p desde el menor:
    (HACER EN FACTIBILIDAD) Ver factibilidad de caja por resistencia a comprension + headspace
    Asignar a p el tipo de caja con menor costo considerando el cálculo de descuentos sumando los volúmenes de producto
    Recalculo descuentos

Con los descuentos finales, vuelvo a calcular cada costo de producto

#### **Greedy 2: Elección por mayor volumen**

Greedy (2)

Ordenamos los productos de menor a mayor según la cantidad de tipos de cajas asignables.
Iteramos los productos p desde el menor:
    (HACER EN FACTIBILIDAD) Ver factibilidad de caja por resistencia a comprension + headspace
    Asignar a p el tipo de caja con mayor volúmen actualmente

Con los descuentos finales, vuelvo a calcular cada costo de producto

#### **Greedy 3: Ordenamiento de productos según su demanda total**

Greedy (3)

Rehacer Greedy 1 y 2 ordenando los productos de mayor a menor según la demanda.

Partiendo de la solucion, veo si la puedo mejorar

En cada paso:
    Cambio el tipo de caja, y me fijo si con los descuentos puedo tener un menor costo

Restriccion: dim_actual <= dim_producto x 1.1

Headspace (grosor 4.5): dim_actual * 0.9 >= dim_producto

dim_actual >= dim_producto : 0.9
dim_actual <= dim_producto x 1.1

dim_producto x 1.11 <= dim_actual <= dim_producto x 1.1